In [1]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt 
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import RidgeClassifier, LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import BernoulliNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, recall_score,
    precision_score, f1_score, cohen_kappa_score,
    matthews_corrcoef
)
from sklearn.inspection import permutation_importance
from scipy.stats import chi2_contingency
from imblearn.over_sampling import RandomOverSampler
import kagglehub
import shutil
import os
from ucimlrepo import fetch_ucirepo

In [2]:
# Kaggle
dataset_dir = "data"
if len(os.listdir(dataset_dir)) == 0:
    path = kagglehub.dataset_download("redwankarimsony/heart-disease-data")
    shutil.move(path, dataset_dir)

print("dataset downloaded")

# UCI dataset
# heart_disease = fetch_ucirepo(id=45) 
# X = heart_disease.data.features 
# y = heart_disease.data.targets 

dataset downloaded


In [3]:
# uncomment if use data from kaggle
data_path = "data/6/heart_disease_uci.csv"
df = pd.read_csv(data_path, index_col=0)
print(df.columns)

Index(['age', 'sex', 'dataset', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
       'thalch', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num'],
      dtype='object')


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 920 entries, 1 to 920
Data columns (total 15 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       920 non-null    int64  
 1   sex       920 non-null    object 
 2   dataset   920 non-null    object 
 3   cp        920 non-null    object 
 4   trestbps  861 non-null    float64
 5   chol      890 non-null    float64
 6   fbs       830 non-null    object 
 7   restecg   918 non-null    object 
 8   thalch    865 non-null    float64
 9   exang     865 non-null    object 
 10  oldpeak   858 non-null    float64
 11  slope     611 non-null    object 
 12  ca        309 non-null    float64
 13  thal      434 non-null    object 
 14  num       920 non-null    int64  
dtypes: float64(5), int64(2), object(8)
memory usage: 115.0+ KB


In [5]:
df.describe()

,age,trestbps,chol,thalch,oldpeak,ca,num
count,920.000000,861.000000,890.000000,865.000000,858.000000,309.000000,920.000000
mean,53.510870,132.132404,199.130337,137.545665,0.878788,0.676375,0.995652
std,9.424685,19.066070,110.780810,25.926276,1.091226,0.935653,1.142693
min,28.000000,0.000000,0.000000,60.000000,-2.600000,0.000000,0.000000
25%,47.000000,120.000000,175.000000,120.000000,0.000000,0.000000,0.000000
50%,54.000000,130.000000,223.000000,140.000000,0.500000,0.000000,1.000000
75%,60.000000,140.000000,268.000000,157.000000,1.500000,1.000000,2.000000
max,77.000000,200.000000,603.000000,202.000000,6.200000,3.000000,4.000000


In [6]:
df.isna().sum()

age           0
sex           0
dataset       0
cp            0
trestbps     59
chol         30
fbs          90
restecg       2
thalch       55
exang        55
oldpeak      62
slope       309
ca          611
thal        486
num           0
dtype: int64

In [7]:
df['ca'] = df['ca'].astype(str)
y = df['num']
X = df.drop(columns=['num', 'dataset'])
numerical_feature = X.select_dtypes(include=["int64", "float64"]).columns.to_list()
categorical_feature = X.select_dtypes(exclude=["int64", "float64"]).columns.to_list()
X[categorical_feature] = X[categorical_feature].astype(str)

In [8]:
def transform_to_biner(x):
    if x != 0:
        x = 1
    return x

y = y.apply(transform_to_biner)

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [10]:
preprocessor = ColumnTransformer(
    transformers=[
        ("numerical", Pipeline([
            ('imputer', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler())
        ]), numerical_feature),
        ("categorical", Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
            ('encoder', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_feature)
    ]
)

In [11]:
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', KNeighborsClassifier())
])

In [12]:
model.fit(X_train, y_train)

result = permutation_importance(
    model, X_test, y_test, scoring="roc_auc", n_repeats=30, random_state=42
)

In [14]:
# 1. Membuat DataFrame untuk menampung nama fitur dan nilai importance-nya
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': result.importances_mean,
    'Std': result.importances_std # Standar deviasi sebagai pengukur ketidakpastian
})

# 2. Mengurutkan berdasarkan nilai Importance dari yang tertinggi ke terendah (descending)
feature_importance = feature_importance.sort_values(by='Importance', ascending=False).reset_index(drop=True)

# 3. Menampilkan ranking dalam bentuk tabel
print("Ranking Fitur berdasarkan Permutation Importance:")
display(feature_importance)


Ranking Fitur berdasarkan Permutation Importance:


,Feature,Importance,Std
0,cp,0.041173,0.009646
1,exang,0.025587,0.010230
2,age,0.024115,0.007427
3,oldpeak,0.022180,0.010504
4,chol,0.020546,0.007243
5,thal,0.018122,0.006224
6,thalch,0.015443,0.007751
7,ca,0.014435,0.006397
8,trestbps,0.011811,0.006495
9,slope,0.010943,0.008929
